In [ ]:
import os
import torch
from torch_geometric.data import Data, Dataset
import torch.nn.functional as F
import torch.nn as nn
from torch.nn import Linear, ReLU, Dropout,LayerNorm

from torch_geometric.loader import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.metrics import brier_score_loss

from scipy.io import loadmat
import numpy as np
import pandas as pd
from torch_geometric.utils import scatter
import torch_geometric.nn as pyg_nn
import torch_geometric.utils as pyg_utils
import shap

import random


seed = 42  # or any fixed number
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [ ]:
class PatientGraphDataset(Dataset):
    def __init__(self, folder):
        super().__init__()
        self.folder = folder
        self.files = sorted([f for f in os.listdir(folder) if f.endswith('.mat')])


    def len(self):
        return len(self.files)

    def get(self, idx):
        file = self.files[idx]
        mat = loadmat(os.path.join(self.folder, file))
        try:
            patient_id_str = file.split('_')[0] 
        except IndexError:
            patient_id_str = f"Unknown_{idx}" # Fallback
        # Load GC matrices
        gc1, gc2, gc3, gc4, gc5, gc6, gc7 = mat['C1'], mat['C2'], mat['C3'], mat['C4'], mat['C5'], mat['C6'],mat['C7']

        th= 0.5
        mask1 = (gc1 > th) | (gc1 < -th)
        mask2 = (gc2 > th) | (gc2 < -th)
        mask3 = (gc3 > th) | (gc3 < -th)
        mask4 = (gc4 > th) | (gc4 < -th)
        mask5 = (gc5 > th) | (gc5 < -th)
        mask6 = (gc6 > th) | (gc6 < -th)
        mask7 = (gc7 > th) | (gc7 < -th)
        final_mask = mask1 | mask2 | mask3 | mask4 | mask5 | mask6 | mask7 

        src, tgt = np.where(final_mask)

        # Extract edge values using common indices
        edge_vals1 = gc1[src, tgt].reshape(-1, 1)
        edge_vals2 = gc2[src, tgt].reshape(-1, 1)
        edge_vals3 = gc3[src, tgt].reshape(-1, 1)
        edge_vals4 = gc4[src, tgt].reshape(-1, 1)
        edge_vals5 = gc5[src, tgt].reshape(-1, 1)
        edge_vals6 = gc6[src, tgt].reshape(-1, 1)
        edge_vals7 = gc7[src, tgt].reshape(-1, 1)


        # Concatenate edge attributes along feature dimension
        edge_attr = np.concatenate([edge_vals1, edge_vals2, edge_vals3, edge_vals4, edge_vals5, edge_vals6, edge_vals7], axis=1)  # [E, 3]

        edge_index = torch.tensor([src, tgt], dtype=torch.long)
        edge_attr = torch.tensor(edge_attr, dtype=torch.float)

        features = mat['features']

        new_feature = features[:, [0,1,2,3,4,5,8]] 

        x = torch.tensor(new_feature, dtype=torch.float)

        y_node = torch.tensor(mat['y_node'].flatten(), dtype=torch.float)
        y_graph = torch.tensor([mat['y_fcd'].item()], dtype=torch.float)
        
        graph_feats = mat['graph_feats']
        age =  torch.tensor(float(graph_feats[0,0]), dtype=torch.float)

        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y_node=y_node, y_graph=y_graph, age=age, patient_id=patient_id_str)

In [ ]:
class SimpleEdgeAwareNN(nn.Module):
    def __init__(self, in_channels, edge_dim, hidden_channels):
        super().__init__()
        self.node_encoder = nn.Linear(in_channels, hidden_channels)
        self.edge_encoder = nn.Linear(edge_dim, hidden_channels)
        self.norm = nn.LayerNorm(hidden_channels)
        self.dropout = nn.Dropout(p=0.1)

        self.mlp = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1)
        )

        self.graph_classifier = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1)
        )

        self.fusion = nn.Sequential(
            nn.Linear(2 * hidden_channels, hidden_channels),
            nn.ReLU()
        )

        self.edge_update = nn.Sequential(
            nn.Linear(2 * hidden_channels + hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels)
        )

        self.att_mlp = nn.Sequential(
            nn.Linear(3 * hidden_channels, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 1)
        )


    def forward(self, data):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr

        # Encode and normalize node and edge features
        x = self.norm(self.node_encoder(x))  # [N, hidden]
        edge_attr = self.norm(self.edge_encoder(edge_attr))  # [E, hidden]


        # Edge-aware attention mechanism
        src_x = x[edge_index[0]]  # [E, hidden]
        tgt_x = x[edge_index[1]]  # [E, hidden]

        edge_input = torch.cat([src_x, tgt_x, edge_attr], dim=1)  # [E, 3*hidden]
        edge_attr = self.edge_update(edge_input)  # [E, hidden]


        att_input = torch.cat([src_x, tgt_x, edge_attr], dim=1)  # [E, 3 * hidden]
        att_score = self.att_mlp(att_input)  # [E, 1]
        att_score = pyg_utils.softmax(att_score, edge_index[1])  # softmax per target

        # Weighted edge aggregation (e.g., like attention-weighted MPNN)
        edge_messages = att_score * edge_attr  # [E, hidden]

        agg_msg = torch.zeros_like(x)
        agg_msg = agg_msg.index_add(0, edge_index[1], edge_messages)  # [N, hidden]


        # Concatenate node features with aggregated edge features
        h = self.fusion(torch.cat([x, agg_msg], dim=1))
        h = self.dropout(h)

        node_pred = self.mlp(h).squeeze(-1)
        graph_emb = pyg_nn.global_mean_pool(h, data.batch)
        graph_pred = self.graph_classifier(graph_emb).squeeze(-1)


        return node_pred, graph_pred

In [ ]:
dataset = PatientGraphDataset("data folder")

avg_acc, avg_sen, avg_pre, avg_f1, avg_spe = [], [], [], [], []
all_results = []
all_node_probs = []
all_node_labels = []

all_graph_probs = []
all_graph_labels = []

patient_outcomes = {}

for i in range(len(dataset)):
    test_data = dataset[i]
    train_data = [dataset[j] for j in range(len(dataset)) if j != i]

    train_loader = DataLoader(train_data, batch_size=len(train_data), shuffle=True)
    test_loader = DataLoader([test_data], batch_size=1)

    model = SimpleEdgeAwareNN(in_channels=7, edge_dim=7, hidden_channels=32)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

    # ── Node pos_weight: count across all training graphs for this fold ───
    n_node_pos = sum(
        dataset[j].y_node.sum().item() 
        for j in range(len(dataset)) if j != i
    )
    n_node_neg = sum(
        (dataset[j].y_node == 0).sum().item() 
        for j in range(len(dataset)) if j != i
    )
    node_pos_weight = torch.tensor(n_node_neg / max(n_node_pos, 1))

    # ──  Graph pos_weight: Compute pos_weight once per fold ───────────
    n_train_pos      = sum(dataset[j].y_graph.item() for j in range(len(dataset)) if j != i)
    n_train_neg      = (len(dataset) - 1) - n_train_pos
    graph_pos_weight = torch.tensor(n_train_neg / max(n_train_pos, 1))


    train_losses = []
    alpha = 0.5
    
    for epoch in range(50):
        model.train()
        total_loss = 0
        for data in train_loader:
            optimizer.zero_grad()
            node_pred, graph_pred = model(data)

            node_loss = F.binary_cross_entropy_with_logits(node_pred, data.y_node, pos_weight=node_pos_weight)
            graph_loss = F.binary_cross_entropy_with_logits(graph_pred, data.y_graph, pos_weight=graph_pos_weight)

            loss= node_loss
            loss = alpha * node_loss + (1 - alpha) * graph_loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        train_losses.append(avg_loss)


    # Evaluation
    model.eval()
    with torch.no_grad():
        for data in test_loader:
            node_pred, graph_pred = model(data)

            probs = torch.sigmoid(node_pred)
            probs2 = torch.sigmoid(graph_pred)

            node_labels = data.y_node.cpu().numpy()
            graph_label = data.y_graph.cpu().numpy()

            all_node_probs.extend(probs)
            all_node_labels.extend(node_labels)

            all_graph_probs.extend(probs2)
            all_graph_labels.extend(graph_label)

            node_pred_bin = (probs > 0.5).float()   
            graph_pred_bin = (probs2 > 0.5).float()

            node_labels_np = node_labels
            node_pred_np = node_pred_bin.cpu().numpy()

            node_outcomes = []

            for n in range(len(node_labels_np)):
                true_label = int(node_labels_np[n])
                pred_label = int(node_pred_np[n])

                if true_label == 1 and pred_label == 1:
                    outcome = "TP"
                elif true_label == 0 and pred_label == 0:
                    outcome = "TN"
                elif true_label == 0 and pred_label == 1:
                    outcome = "FP"
                else:
                    outcome = "FN"

                node_outcomes.append(outcome)


            acc = accuracy_score(data.y_node.cpu(), node_pred_bin.cpu())
            sens =  recall_score(data.y_node.cpu(), node_pred_bin.cpu(), pos_label=1, zero_division=0)
            spe= recall_score(data.y_node.cpu(), node_pred_bin.cpu(), pos_label=0, zero_division=0)
            prec = precision_score(data.y_node.cpu(), node_pred_bin.cpu(), average='macro', zero_division=0)
            f1 = f1_score(data.y_node.cpu(), node_pred_bin.cpu(), average='macro', zero_division=0)
            real_fcd=data.y_graph.item()
            predict_fcd= graph_pred_bin.item()

            avg_acc.append(acc)
            avg_sen.append(sens)
            avg_spe.append(spe)
            avg_pre.append(prec)
            avg_f1.append(f1)
            

            fcd='false'
            if real_fcd==predict_fcd:
                fcd='true'

            all_results.append({
                'last loss': avg_loss,
                'test_patient': i,
                'node_accuracy': acc,
                'node_sensitivity': sens,
                'node_specificity': spe,
                'node_precision': prec,
                'node_f1': f1,
                'real FCD': real_fcd,
                'predicted FCD': predict_fcd,
                'prediction score': probs2,
                'fcd classification':fcd
            })


# Print final results
for r in all_results:
    print(r)




print("\nAverages:")
print( sum(avg_acc)/len(avg_acc)* 100)
print( sum(avg_sen)/len(avg_sen)* 100)
print( sum(avg_spe)/len(avg_spe)* 100)
print( sum(avg_pre)/len(avg_pre)* 100)
print( sum(avg_f1)/len(avg_f1))
print(" ")

node_brier = brier_score_loss(all_node_labels, all_node_probs)
graph_brier = brier_score_loss(all_graph_labels, all_graph_probs)

print("\nBrier Scores:")
print("SOZ Node Classification:", node_brier)
print("FCD Graph Classification:", graph_brier)

In [ ]:
# --------------------------------------------------
# Feature importance SHAP for Graph classification
# --------------------------------------------------

all_shap_values = []
all_inputs = []
all_patient_ids = []

tp_shap_values_fcd = []
tn_shap_values_fcd = []

# Assuming the feature names correspond exactly to the concatenation:

feature_names = ['HFO', 'IED', 'Anomaly', 'HFO-IED', 'HFO-HFO', 'IED-IED', 'PAC',
    'Theta','Alpha','Beta','Low gamma','High gamma','Ripple','Fast Ripple'
] # Corrected names to reflect the feature source

device = next(model.parameters()).device # Get device from model
model.eval()


background_data = []
for graph in dataset:
    node_avg = graph.x.numpy().mean(axis=0)
    edge_avg = graph.edge_attr.numpy().mean(axis=0)
    combined = np.concatenate([node_avg, edge_avg])
    background_data.append(combined)

background_data = np.vstack(background_data)

N_background_samples = min(50, len(background_data))
background = background_data[np.random.choice(len(background_data), N_background_samples, replace=False)]

# -------------------------------
# 2) Prediction function 
# -------------------------------
def predict_graph_with_avg_features(input_numpy, original_graph):
    """
    Predicts graph classification for a batch of average feature vectors.
    
    input_numpy: SHAP's input (a batch of perturbed average feature vectors).
    original_graph: The specific graph structure to apply features to.
    """
    preds = []
    
    g = original_graph.clone().to(device)

    for row in input_numpy:
        
        if row.shape[0] != 14:
            raise ValueError(f"Input row size must be 14, got {row.shape[0]}")

        # Separate the perturbed average features
        node_avg_feat = torch.tensor(row[:7], dtype=torch.float).to(device)
        edge_avg_feat = torch.tensor(row[7:], dtype=torch.float).to(device)
        
        g.x = node_avg_feat.repeat(g.num_nodes, 1)
        g.edge_attr = edge_avg_feat.repeat(g.num_edges, 1)

        with torch.no_grad():
            g.batch = torch.tensor([0] * g.num_nodes, dtype=torch.long).to(device)
            _, graph_pred = model(g)
            
            preds.append(graph_pred.cpu().item())

    return np.array(preds).reshape(-1, 1)


# -------------------------------
# 3) Loop and Run Kernel SHAP
# -------------------------------
for patient_id, graph in enumerate(dataset):
    
    def patient_predict_fn(input_numpy):
        return predict_graph_with_avg_features(input_numpy, original_graph=graph)

    explainer = shap.KernelExplainer(patient_predict_fn, background)

    node_avg = graph.x.numpy().mean(axis=0)
    edge_avg = graph.edge_attr.numpy().mean(axis=0)
    test_input = np.concatenate([node_avg, edge_avg]).reshape(1, -1)
    
    shap_vals = explainer.shap_values(test_input, nsamples=100) 

    shap_vec = shap_vals[0].reshape(-1) 
    all_shap_values.append(shap_vec)
    all_inputs.append(test_input[0])
    all_patient_ids.append(patient_id)

    true_label = graph.y_graph.item()

    with torch.no_grad():
        _, graph_logits = model(graph)
        prob = torch.sigmoid(graph_logits).item()

    pred_label = int(prob > 0.7)

    if true_label == 1 and pred_label == 1:
        tp_shap_values_fcd.append(shap_vals[0].reshape(-1))

    elif true_label == 0 and pred_label == 0:
        tn_shap_values_fcd.append(shap_vals[0].reshape(-1))

    base = explainer.expected_value[0]

    explanation = shap.Explanation(
        values=shap_vec,
        base_values=base,          
        data=test_input[0],
        feature_names=feature_names
    )
    fig = plt.figure()
    shap.plots.waterfall(explanation, max_display=20)
   


# -------------------------------
# 4) Combined SHAP summary plot (all patients)
# -------------------------------
all_shap_values = np.vstack(all_shap_values)
all_inputs = np.vstack(all_inputs)
fig= plt.figure(figsize=(10, 6))
shap.summary_plot(
    all_shap_values,
    all_inputs,
    feature_names=feature_names,
    show=False
)
plt.title("All Patients — SHAP Summary (Aggregated Features)")
plt.tight_layout()
plt.show()


# Calculate mean absolute SHAP value per feature
mean_abs_shap = np.abs(all_shap_values).mean(axis=0)
print(mean_abs_shap)


# Sort by importance
sorted_idx = np.argsort(mean_abs_shap)[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]
sorted_shap_values = mean_abs_shap[sorted_idx]

# Plot
fig= plt.figure(figsize=(8, 5))
plt.barh(sorted_features, sorted_shap_values)
plt.xlabel("Mean |SHAP value|")
plt.title("Global Feature Importance (SHAP Bar Plot)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()




In [ ]:
# --------------------------------------------------
# Feature importance SHAP for Node classification
# --------------------------------------------------

tp_shap_values = []
tn_shap_values = []


all_patient_ids = [graph.patient_id for graph in dataset]
unique_patient_ids = sorted(list(set(all_patient_ids)))

patient_shap_data = {
    p_id: {'shap_values': [], 'inputs': []} 
    for p_id in unique_patient_ids
}

all_shap_values = []
all_inputs = []


###############################################################################
# 1. Extract edges connected to each node (function remains the same)
###############################################################################
def get_edges_for_node(graph, node_idx):
    src, dst = graph.edge_index
    mask = (src == node_idx) | (dst == node_idx) 
    return mask.nonzero(as_tuple=False).view(-1)


###############################################################################
# 2. SHAP predictor (Class remains the same)
###############################################################################
class SHAPPredictor:
    def __init__(self, model, graph, node_idx, connected_edge_ids):
        self.model = model.eval()
        self.graph = graph
        self.node_idx = node_idx
        self.connected_edge_ids = connected_edge_ids
        self.device = next(model.parameters()).device if list(model.parameters()) else torch.device("cpu")

    def __call__(self, inputs):
        outputs = []
        data = self.graph.clone().to(self.device)
        inputs_tensor = torch.tensor(inputs, dtype=torch.float32).to(self.device)

        for row in inputs_tensor:
            node_feats = row[:7]
            avg_edge_feats = row[7:]
            
            temp_x = data.x.clone()
            temp_x[self.node_idx] = node_feats
            temp_edge_attr = data.edge_attr.clone()
            temp_edge_attr[self.connected_edge_ids] = avg_edge_feats
            data.x = temp_x
            data.edge_attr = temp_edge_attr

            with torch.no_grad():
                node_logits, _ = self.model(data)
                if node_logits.dim() > 1:
                    logit = node_logits[self.node_idx, 0]
                else:
                    logit = node_logits[self.node_idx]
                prob = torch.sigmoid(logit).item()
                outputs.append(prob)
                
        return np.array(outputs).reshape(-1, 1)


###############################################################################
# 3. Main loop over graphs and nodes (Processing and Aggregation)
###############################################################################

dataset_list = list(dataset) 
SAMPLE_SIZE = 5

for graph_idx, graph in enumerate(dataset_list): 

    current_patient_id = graph.patient_id 

    if len(dataset_list) >= SAMPLE_SIZE:
        background_graphs = random.sample(dataset_list, SAMPLE_SIZE)
    else:
        background_graphs = dataset_list 
    
    background_data_list = []
    for g_b in background_graphs: 
        for nid in range(min(g_b.num_nodes, 5)): 
            eids = get_edges_for_node(g_b, nid)
            if eids.numel() == 0:
                continue
            
            node_b = g_b.x[nid].cpu().numpy()
            edge_b = g_b.edge_attr[eids].mean(dim=0).cpu().numpy()
            background_data_list.append(np.concatenate([node_b, edge_b]))
    
    if len(background_data_list) < 10:
        continue 
        
    background = np.vstack(background_data_list)
    

    for node_idx in range(graph.num_nodes):
        connected_edge_ids = get_edges_for_node(graph, node_idx).view(-1)
        if connected_edge_ids.numel() == 0:
            continue

        mean_edge_feat = graph.edge_attr[connected_edge_ids].mean(dim=0).cpu().numpy()
        node_feat = graph.x[node_idx].cpu().numpy()
        test_input = np.concatenate([node_feat, mean_edge_feat]).reshape(1, -1)
        
        predictor = SHAPPredictor(model, graph, node_idx, connected_edge_ids)
        explainer = shap.KernelExplainer(predictor, background)
        shap_vals = explainer.shap_values(test_input, nsamples=100)

    
        patient_shap_data[current_patient_id]['shap_values'].append(shap_vals[0].reshape(-1))
        patient_shap_data[current_patient_id]['inputs'].append(test_input[0])
        

        all_shap_values.append(shap_vals[0].reshape(-1))
        all_inputs.append(test_input[0])

        true_label = graph.y_node[node_idx].item()

        with torch.no_grad():
            node_logits, _ = model(graph)
            prob = torch.sigmoid(node_logits[node_idx]).item()

        pred_label = int(prob > 0.55)

        if true_label == 1 and pred_label == 1:
            tp_shap_values.append(shap_vals[0].reshape(-1))

        elif true_label == 0 and pred_label == 0:
            tn_shap_values.append(shap_vals[0].reshape(-1))

    base = explainer.expected_value[0]

    explanation = shap.Explanation(
        values=shap_vals[0].reshape(-1),
        base_values=base,         
        data=test_input[0],
        feature_names=feature_names
    )
    fig = plt.figure()
    shap.plots.waterfall(explanation, max_display=20)
   
   


###############################################################################
# 4. Final Plotting (Overall and Per Patient)
###############################################################################

if all_shap_values:
    all_shap_values_st = np.vstack(all_shap_values)
    all_inputs = np.vstack(all_inputs)

    print("\n--- Generating Global SHAP Summary Plot (All Patients) ---")
    fig=plt.figure(figsize=(10, 6))
    shap.summary_plot(all_shap_values_st, all_inputs, feature_names=feature_names, title="Node Classification SHAP - ALL PATIENTS")
   


# --- Per Patient Plots ---
print("\n--- Generating SHAP Summary Plots Per Patient ---")
for p_id, data in patient_shap_data.items():
    if not data['shap_values']:
        continue 
        
    p_shap_vals = np.vstack(data['shap_values'])
    p_inputs = np.vstack(data['inputs'])

    fig=plt.figure(figsize=(10, 6))
    shap.summary_plot(p_shap_vals, p_inputs, feature_names=feature_names,title="SHAP Summary Plot - Patient {p_id}")
    
mean_abs_shap = np.mean(np.abs(all_shap_values), axis=0)
print(mean_abs_shap)

std_abs_shap = np.std(np.abs(all_shap_values), axis=0)

sorted_idx = np.argsort(mean_abs_shap)[::-1]
sorted_shap = mean_abs_shap[sorted_idx]
sorted_std = std_abs_shap[sorted_idx]
sorted_features = [feature_names[i] for i in sorted_idx]

plt.figure(figsize=(10, 6))
plt.barh(sorted_features, sorted_shap)
plt.xlabel("Mean |SHAP value|")
plt.title("Global Feature Importance with Variation (SHAP)")
plt.gca().invert_yaxis()  
plt.tight_layout()
plt.show()

